In [ ]:
#List of videos per patient
import pandas as pd


data = [
    {'Patient': 1, 'Files': ['20190830T115515', '20190830T115602', '20190830T115644']},
    {'Patient': 2,  'Files': ['20190904T101559']},
    {'Patient': 3, 'Files': ['20190906T105145', '20190906T105237']},
    {'Patient': 4, 'Files': ['20190909T155747']},
    {'Patient': 5, 'Files': ['20190909T161453', '20190909T161601']},
    {'Patient': 6, 'Files': ['20190911T104437', '20190911T105058']},
    {'Patient': 7, 'Files': ['20190911T111121']},
    {'Patient': 8, 'Files': ['20190916T104520']},
    {'Patient': 9, 'Files': ['20190916T105526', '20190916T105641']},
    {'Patient': 10, 'Files': ['20190916T110257']},
    {'Patient': 11, 'Files': ['20190916T112312']},
    {'Patient': 12, 'Files': ['20190918T115054']},
    {'Patient': 13, 'Files': ['20190918T120011','20190918T120628','20190918T120708']},
    {'Patient': 14, 'Files': ['20190918T123342','20190918T123437']},
    {'Patient': 15, 'Files': ['20190922T101601']},
    {'Patient': 16, 'Files': ['20190923T173644']},
    {'Patient': 17, 'Files': ['20190923T175155']},
    {'Patient': 18, 'Files': ['20190930T110010']},
    {'Patient': 19, 'Files': ['20191003T173034']},
    {'Patient': 20, 'Files': ['20191008T112326']},
    {'Patient': 21, 'Files': ['20191008T114159', '20191008T114249']},
    {'Patient': 22, 'Files': ['20191026T195815']},
    {'Patient': 23, 'Files': ['20191108T114950']},
    {'Patient': 24, 'Files': ['20191115T105623', '20191115T105730']},
    {'Patient': 25, 'Files': ['20191115T110256']},
    {'Patient': 26, 'Files': ['20191115T112747']},
    {'Patient': 27, 'Files': ['20191115T114514']},
    {'Patient': 28, 'Files': ['20191127T110323', '20191127T110427']},
    {'Patient': 29, 'Files': ['20191127T111516']},
    {'Patient': 30, 'Files': ['20191127T112700','20191127T112757']},
    {'Patient': 31, 'Files': ['20191127T113658','20191127T113821']},
    {'Patient': 32, 'Files': ['20191129T103133','20191129T103822']},
    {'Patient': 33, 'Files': ['20191129T105514']},
    {'Patient': 34, 'Files': ['20191129T110558','20191129T110732']},
    {'Patient': 35, 'Files': ['20191203T105250']},
    {'Patient': 36, 'Files': ['20191203T111627','20191203T111732']},
    {'Patient': 37, 'Files': ['20191205T103749', '20191205T103904' ]},
    {'Patient': 38, 'Files': ['20191208T164549','20191208T165241']},
    {'Patient': 39, 'Files': ['20191208T170945']},
    {'Patient': 40, 'Files': ['20191212T102143']},
    {'Patient': 41, 'Files': ['20191212T103205','20191212T103310']},
    {'Patient': 42, 'Files': ['20191214T100241']},
    {'Patient': 43, 'Files': ['20191214T103803']},
    {'Patient': 44, 'Files': ['20191218T104745', '20191218T104848']},
    {'Patient': 45, 'Files': ['20191218T105735','20191218T105909', '20191218T110113']},
    {'Patient': 46, 'Files': ['20191220T102712']},
    {'Patient': 47, 'Files': ['20191220T104055']},
    {'Patient': 48, 'Files': ['20191220T112002','20191220T112127']},
    {'Patient': 49, 'Files': ['20191220T113126','20191220T113230']},
    {'Patient': 50, 'Files': ['20200103T102623','20200103T102728']},
    {'Patient': 51, 'Files': ['20200103T104919','20200103T105033']}
]

# Convert the list of dictionaries to a pandas DataFrame, converting lists to strings
for item in data:
    if isinstance(item['Files'], list):
        item['Files'] = ';'.join(item['Files'])

df = pd.DataFrame(data)

# Save the DataFrame to a CSV file, overwriting the existing file
output_csv = r'C:\Users\cinth\Documentos\ams\data_science\actual_thesis\codes\MedSAM_Universeg_2024\datasets\data\patients_data.csv'
df.to_csv(output_csv, index=False)

print(f"Data has been successfully written to {output_csv}")


In [ ]:
import os
import random
import shutil
import pandas as pd

def ensure_directory_exists(path):
    try:
        os.makedirs(path, exist_ok=True)
    except Exception as e:
        print(f"Error creating directory {path}: {e}")
        raise

def copy_patient_data(patient_list, patient_data, dest_image_folder, dest_mask_folder, image_patient_list, partition):
    for patient in patient_list:
        try:
            patient_videos = patient_data[patient_data['Patient'] == patient]['Files'].values[0].split(';')
        except KeyError:
            print(f"Patient {patient} not found in patient data.")
            continue

        for video_folder in patient_videos:
            video_folder_path = os.path.join(input_folder, video_folder)
            if os.path.isdir(video_folder_path):
                mask_enhance_folder = os.path.join(video_folder_path, 'mask_enhance')
                if os.path.exists(mask_enhance_folder):
                    try:
                        images = os.listdir(os.path.join(video_folder_path, 'image'))
                        masks = os.listdir(mask_enhance_folder)
                    except FileNotFoundError as e:
                        print(f"Error: {e}")
                        continue
                    
                    if len(images) != len(masks):
                        print(f"Mismatch in number of images and masks for patient {patient} in folder {video_folder_path}.")
                        continue
                    
                    # Sorting the images and masks to maintain order
                    images.sort()
                    masks.sort()

                    copy_images_and_masks(images, masks,
                                          os.path.join(video_folder_path, 'image'),
                                          mask_enhance_folder,
                                          dest_image_folder,
                                          dest_mask_folder,
                                          patient,
                                          partition,
                                          image_patient_list)
                else:
                    print(f"Mask enhance folder not found for patient {patient} in {video_folder_path}.")
            else:
                print(f"Video folder not found: {video_folder_path}.")

def copy_images_and_masks(images, masks, src_image_folder, src_mask_folder, dest_image_folder, dest_mask_folder, patient, partition, image_patient_list):
    for image_name, mask_name in zip(images, masks):
        src_image_path = os.path.join(src_image_folder, image_name)
        src_mask_path = os.path.join(src_mask_folder, mask_name)
        dest_image_path = os.path.join(dest_image_folder, image_name)
        dest_mask_path = os.path.join(dest_mask_folder, mask_name)
        try:
            shutil.copyfile(src_image_path, dest_image_path)
            shutil.copyfile(src_mask_path, dest_mask_path)
            image_patient_list.append({'Image': image_name, 'Patient': patient, 'Partition': partition})
        except FileNotFoundError as e:
            print(f"Error copying files: {e}")
        except Exception as e:
            print(f"Unexpected error: {e}")

def save_image_patient_csv(image_patient_list, output_csv_path):
    df = pd.DataFrame(image_patient_list)
    df.to_csv(output_csv_path, index=False)
    print(f"Image-patient mapping has been successfully written to {output_csv_path}")

# Define input and output folders
input_folder = r''
output_folder_images = r''
output_folder_masks = r''
ensure_directory_exists(output_folder_images)
ensure_directory_exists(output_folder_masks)

# Define proportions for training, validation, and testing sets (as fractions)
train_ratio = 0.6
val_ratio = 0.2
test_ratio = 0.2

# Validate that the ratios sum to 1
assert train_ratio + val_ratio + test_ratio == 1.0, "The sum of train, val, and test ratios must be 1.0"

# Create output folders for training, validation, and testing sets
train_folder_images = os.path.join(output_folder_images, 'train')
val_folder_images = os.path.join(output_folder_images, 'val')
test_folder_images = os.path.join(output_folder_images, 'test')
train_folder_masks = os.path.join(output_folder_masks, 'train')
val_folder_masks = os.path.join(output_folder_masks, 'val')
test_folder_masks = os.path.join(output_folder_masks, 'test')
ensure_directory_exists(train_folder_images)
ensure_directory_exists(val_folder_images)
ensure_directory_exists(test_folder_images)
ensure_directory_exists(train_folder_masks)
ensure_directory_exists(val_folder_masks)
ensure_directory_exists(test_folder_masks)

# Read patient data
try:
    patient_data = pd.read_csv(r'C:\Users\cinth\Documentos\ams\data_science\actual_thesis\codes\MedSAM_Universeg_2024\datasets\data\patients_data.csv')
except FileNotFoundError:
    print("Patient data file not found.")
    raise
except pd.errors.EmptyDataError:
    print("Patient data file is empty.")
    raise
except Exception as e:
    print(f"Error reading patient data file: {e}")
    raise

# Split patients with a fixed random seed for reproducibility
patients = patient_data['Patient'].unique()
random.seed(0)
random.shuffle(patients)

train_size = int(train_ratio * len(patients))
val_size = int(val_ratio * len(patients))
test_size = len(patients) - train_size - val_size

train_patients = patients[:train_size]
val_patients = patients[train_size:train_size + val_size]
test_patients = patients[train_size + val_size:]

# List to store image-patient mapping
image_patient_list = []

# Process the splits
copy_patient_data(train_patients, patient_data, train_folder_images, train_folder_masks, image_patient_list, 'train')
copy_patient_data(val_patients, patient_data, val_folder_images, val_folder_masks, image_patient_list, 'val')
copy_patient_data(test_patients, patient_data, test_folder_images, test_folder_masks, image_patient_list, 'test')

# Save the image-patient mapping to a CSV file
output_csv_path = r'C:\Users\cinth\Documentos\ams\data_science\actual_thesis\codes\MedSAM_Universeg_2024\datasets\data\image_patient_mapping.csv'
save_image_patient_csv(image_patient_list, output_csv_path)

print("Dataset partitioning completed.")
